# Φάση Α: Data Preprocessing

**Υπεύθυνος:** Data Engineer

**Βήματα:**
1. Φόρτωση δεδομένων
2. Καθαρισμός (missing values, outliers)
3. Feature engineering
4. Encoding κατηγορικών μεταβλητών
5. Train/test split
6. Handling class imbalance

Πριν αρχίσει οποιαδήποτε διεργασία πρέπει να setαριστούν κάποια βασικά πράγματα.


1

In [ ]:
#
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand

# Αρχικοποίηση του Spark Session
spark = SparkSession.builder \
    .appName("StrokePrediction_DataEngineering") \
    .master("local[*]") \
    .getOrCreate()

print("Spark Session αρχικοποιήθηκε επιτυχώς!")

2

In [ ]:
# Ορισμός του μονοπατιού του αρχείου (προσάρμοσε το όνομα αν διαφέρει)
file_path = "healthcare-dataset-stroke-data.csv"

# Ανάγνωση του CSV με αυτόματη ανίχνευση Schema και Headers
df_raw = spark.read.csv(file_path, header=True, inferSchema=True)

# Αφαίρεση της στήλης ID
df_cleaned = df_raw.drop("id")

print(f"Συνολικές γραμμές Bronze Layer: {df_cleaned.count()}")
df_cleaned.printSchema()

3

In [ ]:
# Χωρισμός σε Train και Test (80-20) με seed για σταθερότητα
train_initial, test_df = df_cleaned.randomSplit([0.8, 0.2], seed=42)

print(f"Αρχικό Train Set: {train_initial.count()} γραμμές")
print(f"Καθαρό Test Set (Ανέγγιχτο): {test_df.count()} γραμμές")

# Έλεγχος της ανισορροπίας (Class Imbalance) στο Train
print("\nΚατανομή κλάσεων στο αρχικό Train Set:")
train_initial.groupBy("stroke").count().show()


4

In [ ]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

categorical_cols = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]

# Δημιουργία Indexers που μετατρέπουν τα κείμενα σε αριθμητικά indexes (π.χ. "Male" -> 0.0)
indexers = [StringIndexer(inputCol=col_name, outputCol=f"{col_name}_index", handleInvalid="keep") for col_name in categorical_cols]

# Εκτελούμε αυτό το μίνι-pipeline στο Train για να πάρουμε αριθμητικές στήλες
indexer_pipeline = Pipeline(stages=indexers)
indexer_model = indexer_pipeline.fit(train_initial)

train_indexed = indexer_model.transform(train_initial)
# Κάνουμε το ίδιο και για το Test για να είναι έτοιμο για μετά
test_indexed = indexer_model.transform(test_df)

5

In [ ]:
import pandas as pd
from imblearn.over_sampling import SMOTE

# Επιλέγουμε τις αριθμητικές στήλες και τα νέα indexes που φτιάξαμε
numeric_cols = ["age", "hypertension", "heart_disease", "avg_glucose_level", "bmi"]
indexed_cols = [f"{col_name}_index" for col_name in categorical_cols]
features_to_smote = numeric_cols + indexed_cols

# Μετατροπή σε Pandas
train_pdf = train_indexed.select(features_to_smote + ["stroke"]).toPandas()

# Διαχείριση τυχόν Null τιμών στο BMI πριν το SMOTE (το SMOTE δεν δέχεται NaN)
# Γεμίζουμε προσωρινά με τη διάμεσο του Train
train_pdf["bmi"] = train_pdf["bmi"].fillna(train_pdf["bmi"].median())

X_train = train_pdf[features_to_smote]
y_train = train_pdf["stroke"]

# Εκτέλεση SMOTE
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# Ανασύσταση του ισορροπημένου DataFrame
balanced_train_pdf = pd.DataFrame(X_train_res, columns=features_to_smote)
balanced_train_pdf["stroke"] = y_train_res

# Επιστροφή στον Spark (Silver Layer)
train_balanced_spark = spark.createDataFrame(balanced_train_pdf)

print(f"Νέο Ισορροπημένο Train Set με SMOTE: {train_balanced_spark.count()} γραμμές")
train_balanced_spark.groupBy("stroke").count().show()

6

In [ ]:
from pyspark.ml.feature import OneHotEncoder, VectorAssembler, StandardScaler

# OneHotEncoding στις αριθμητικοποιημένες κατηγορικές στήλες
encoders = [OneHotEncoder(inputCol=f"{col_name}_index", outputCol=f"{col_name}_vec") for col_name in categorical_cols]

# Συγκέντρωση όλων των features (Vectors + Αριθμητικά)
assembler_inputs = [f"{col_name}_vec" for col_name in categorical_cols] + numeric_cols
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="assembled_features")

# Scaling (Κανονικοποίηση) για τους αλγορίθμους (π.χ. SVM, Neural Networks)
scaler = StandardScaler(inputCol="assembled_features", outputCol="features", withStd=True, withMean=False)

# Δέσιμο του τελικού Pipeline
final_pipeline = Pipeline(stages=encoders + [assembler, scaler])

# Fit ΜΟΝΟ στο ισορροπημένο Train Set
final_pipeline_model = final_pipeline.fit(train_balanced_spark)

# Παραγωγή του τελικού Gold Layer και για τα δύο σύνολα
train_gold = final_pipeline_model.transform(train_balanced_spark).select("features", "stroke")
test_gold = final_pipeline_model.transform(test_indexed).select("features", "stroke")

print("--- Δείγμα από το τελικό Train Gold Layer (Ισορροπημένο με SMOTE) ---")
train_gold.show(3, truncate=False)

print("\n--- Δείγμα από το τελικό Test Gold Layer (Φυσικό & Ανισόρροπο για σωστό Evaluation) ---")
test_gold.show(3, truncate=False)